In [49]:
import math
import random

In [236]:
class Mathlib:
    @staticmethod
    def clip(val, minVal=0, maxVal=1):
        return(min(maxVal, max(minVal, val)))

    @staticmethod
    def mean(vals):
        return(sum(vals)/len(vals))
    
    @staticmethod
    def argmax(vals):
        maxVal = vals[0]
        maxIndex = 0
        for i, v in enumerate(vals):
            if v > maxVal:
                maxVal = v
                maxIndex = i
        return(maxIndex)
    
    @staticmethod
    def argmin(vals):
        minVal = vals[0]
        minIndex = 0
        for i, v in enumerate(vals):
            if v < minVal:
                minVal = v
                minIndex = i
        return(minIndex)

    @staticmethod
    def dot2Vectors(vector1, vector2):
        out = 0
        for v1, v2 in zip(vector1, vector2):
            out += v1*v2
        return(out)
    
    @staticmethod
    def dotVectorMatrix(vector, matrix):
        out = []
        for row in matrix:
            out.append(Mathlib.dot2Vectors(vector, row))
        return(out)
    
    @staticmethod
    def elementWiseMult(v1, v2):
        result = []
        for i1, i2 in zip(v1, v2):
            result.append(i1*i2)
        return(result)

    @staticmethod
    def normalize(values):
        out = []
        total = sum(values)
        for v in values:
            out.append(v/total)
        return(out)

#### Go to `proofs_math/ActivationFuncs` for all Activation function proofs
<div style="display: flex; align-items: flex-start; gap: 20px;">
  <div>
    <img src="./proofs_math/ActivationFuncs/SoftmaxDerivative.jpg?version=1"
         alt="Softmax Proof"
         width="500">
  </div>

  <div style="display: flex; flex-direction: column; gap: 20px;">
    <img src="./proofs_math/ActivationFuncs/ReLUDerivative.jpg?version=1"
         alt="ReLU Proof"
         width="500">
    <img src="./proofs_math/ActivationFuncs/LossDerivative.jpg?version=1"
         alt="Loss Proof"
         width="500">
  </div>

</div>



In [162]:
class Activation:
    class ReLU:
        """ Returns 0 if values and x if its not """
        @staticmethod
        def forward(val):
            return(max(0, val))    #< by clipping 0, you de-linearize your data
        
        @staticmethod
        def backward(val):
            """ partial derivative of max(x,0) with respect to x """
            return(1 if val > 0 else 0)

    class Pass:
        """ returns x """
        @staticmethod
        def forward(val):
            return(val)

        @staticmethod
        def backward(*args):
            """ derivative of x with respect to x """
            return(1)

    class Softmax:
        """ converts any output to be squashed from 0 to 1 and also had a nice derivative when paired with  Cross-Entropy """
        @staticmethod
        def forward(vals):
            vals = [math.e**val for val in vals]
            vals = Mathlib.normalize(vals)
            return(vals)
        
        @staticmethod
        def backward(vals):
            return(Activation.Softmax.backward_jacobian(vals))

        @staticmethod
        def backward_jacobian(vals):
            """ derivative of e^x/(sum(e^x) for x in vals) with respect to x
            For proof, see image above or in 'proofs_math/ActivationFuncs'"""
            jacobian = []
            for i, val in enumerate(vals):
                row = []
                for j in range(len(vals)):
                    if j == i:
                        row.append(val*(1-val))
                    else:
                        row.append(-val*vals[j])
                jacobian.append(row)
            return(jacobian)
        
        @staticmethod
        def backward_crossEntropy(vals):
            pass

    class ProtectedSoftmax:
        """ Softmax but without any overflow from e^x being too large """
        @staticmethod
        def forward(vals):
            """ softmax, but between 0 and 1 """
            maxVal = max(vals)
            return(Activation.Softmax.forward([val-maxVal for val in vals]))
        
        @staticmethod
        def backward(vals):
            """ derivative of e^x with respect to x as protected softmax is just regular softmax with extra steps to ensure values are normalized """
            return(Activation.Softmax.backward(vals))


In [198]:
class Loss:
    """ Calculates the error of the output (mainly Softmax) using -log(x) """
    @staticmethod
    def forward_single(val):
        return(-math.log(Mathlib.clip(val, 1e-7, 1-1e-7)))

    @staticmethod
    def forward(vals, targets):
        """calculate the cross entropy loss of a softmax output

        Args:
            vals (list): results
            targets (list): target results
        """
        out = 0
        for v, t in zip(vals, targets):
            out += t*Loss.forward_single(v)   #< loss times the weight of the loss, more important = bigger loss
        return(out)                 #< mean of the losses

    
    @staticmethod
    def forward_batch(vals, targets):
        return(Loss.forward(vals, targets)/len(vals))                 #< mean of the losses

    @staticmethod
    def backward(vals, targets):
        return([-t / Mathlib.clip(v, 1e-7, 1-1e-7) for v, t in zip(vals, targets)])


In [180]:
class Accuracy_hard:
    """ calculates how accurate the the result is, for the hard version, it gets an array of index (must be all 0 and a single value has to be 1), and it returns 1 if that most likely output is at the correct index and 0 if its not """
    def __init__(self):
        self._length = 0
        self.correct = 0
        self.accuracy = 0.0

    @staticmethod
    def calc(result, target):
        if type(target) == list:
            target = Mathlib.argmax(target)
        if Mathlib.argmax(result) == target:
            return(1)
        return(0)

    def epoch(self, result, target):
        self.correct += Accuracy_hard.calc(result, target)
        self._length += 1
        self.accuracy = self.correct/self._length
        return(self.accuracy)

class Accuracy_soft:
    """ calculates how accurate the the result is, for the soft version, it get an array of target results, ex: [0.2, 0.8, 0.0, 0.0] and returns a number to quantify how close it is"""
    def __init__(self):
        self._length = 0
        self.correct = 0.0
        self.accuracy = 0.0

    @staticmethod
    def calc(result, target):
        dot = Mathlib.dot2Vectors(result, target)
        norm_r = math.sqrt(sum(r*r for r in result))
        norm_t = math.sqrt(sum(t*t for t in target))
        soft = dot / (norm_r * norm_t)      #< normalize between 0 and 1
        return(soft)

    def epoch(self, result, target):
        soft = Accuracy_soft.calc(result, target)
        self._length += 1
        self.correct += soft
        self.accuracy = self.correct/self._length
        return(self.accuracy)
        

In [227]:
class Neuron:
    """ A single neuron, stores its inputs, weights, bias, and activation function """
    def __init__(self, inputs, weights, bias, activationFunc=Activation.Pass):
        self.inputs = inputs      #< all outputs from previous layer
        self.weights = weights    #< one weight for every input
        self.bias = bias          #< one bias per neuron
        self.activationFunc = activationFunc
        
        if len(self.weights) != len(self.inputs):
            raise Exception("inputs and weight must have the same length!")

    def forward(self):
        """
        Compute the neuron's output.
        Multiply each input by its corresponding weight, sum the
        results, add the bias and then apply the activation function.
        
        multiply the inputs [i1, i2.. in] with weights [w1, w1... wn] to get i1*w1+i2*w2... +in*wn, then add the bias
        """
        
        return(self.activationFunc.forward(Mathlib.dot2Vectors(self.inputs, self.weights)+self.bias))

    def backward(self, output):
        """ Compute gradient 
        since forward is computed as Activation(sum(x1*w1, x2*w2..., bias)),
        the backward for the weights would be computed as [Activation`(...)*sum`(...)*xi] for every element,
        and the backward for the inputs would be computed as [Activation`(...)*sum`(...)*wi] for every element,
        note that sum`(...) is equal to 1 no matter the reference or input.
        """
        activation_dx = self.activationFunc.backward(output)
        inputs_dx = [activation_dx*w for w in self.weights]
        weights_dx = [activation_dx*i for i in self.inputs]
        bias_dx = activation_dx
        return(inputs_dx, weights_dx, bias_dx)


In [226]:
class Layer:
    """A collection of neurons forming a single layer in the network"""
    def __init__(self, inputs, weights, biases,
                 activationFunc=Activation.Pass,   #< Neuron-level activation function
                 *args, **kwargs):
        self.out = None
        self.activationFunc = activationFunc
        self.neurons = self._createNeurons(inputs, weights, biases, activationFunc=activationFunc, *args, **kwargs)  # create neuron objects

    def _createNeurons(self, inputs, weights, biases, activationFunc, *args, **kwargs):
        neurons = []
        for w, b in zip(weights, biases):
            neurons.append(Neuron(inputs, w, b, activationFunc, *args, **kwargs))
        return(neurons)

    def forward(self):
        """Calculate outputs for all neurons in the layer """
        self.out = [neuron.forward() for neuron in self.neurons]    #< saving to use in the backpropagation with chain rule
        return(self.out)
    
    def backward(self, dvalues):
        """Calculate gradients for all neurons in the layer."""
        neuronOutputs = [neuron.backward(dval) for neuron, dval in zip(self.neurons, dvalues)] #< we can calculate these 
        return(neuronOutputs)


In [167]:
class Batch:
    """ Process multiple samples at once in a batch."""
    def __init__(self, inputsBatch, *args, **kwargs):
        self.batch = self._createBatch(inputsBatch, *args, **kwargs)  # create a Layer for each sample

    def _createBatch(self, inputsBatch, *args, **kwargs):
        """Helper method to create a Layer object for each sample in the batch."""
        batch = []
        for inputs in inputsBatch:
            batch.append(Layer(inputs, *args, **kwargs))  # instantiate Layer and add to batch
        return(batch)
    
    def forward(self):
        """Run all layers in the batch and return their outputs."""
        return([layer.forward() for layer in self.batch])
    
    def calcLoss(self, correctIndexes):
        """Compute mean loss across the batch given a list of target indexes."""
        return(Mathlib.mean([self.batch[i].calcLoss(correctIndexes[i]) for i in range(len(self.batch))]))  # average of per-sample losses


In [168]:
class Train:
    """ TBD """
    def __init__(self, weights, biases, learningRate=0.01):
        self.weights = weights
        self.biases = biases
        self.learningRate = learningRate

    def step(self, inputs, correctIndex):
        """Perform a single gradient descent update for one example.

        Args:
            inputs: list of input features for the sample
            correctIndex: index of the true class in the output vector
        """
        layer = Layer(inputs, self.weights, self.biases)
        out = layer.forward()
        for i in range(len(self.weights)) :
            # target probability is 1 for correct class, 0 otherwise (eg: 0 for cat, 0 for dog, 1 for horse)
            target = 0
            if i == correctIndex:
                target = 1
  
            # protectedSoftmax produces an output of e^x/(sum(e^x))
            # the error function is -log(result)
            # the error is -sum(targetX*log(e^x/(sum(e^x)))
            # for target with 0s and 1s [0, 0, 1, 0], the error simplifies to -log(e^x/(sum(e^x))
            # to know if the error has improved, we need to take the derivative of the error
            # the derivative of -log(e^x/(sum(e^x)) simplifies to result-target
            error = out[i] - target
            for j in range(len(self.weights[i])):
                # learning rate determines the size of the change
                # the error determines how far away we are, therefore the size of the update
                # inputs determines the importance of that input
                # an smaller input that has less effect, will be changed less
                self.weights[i][j] -= self.learningRate * error * inputs[j]
            # learning rate determines the size of the change
            # the error determines how far away we are, therefore the size of the update
            # since we update both the weights and biases, we might overcompensate, which is ok because there are multiple iterations
            self.biases[i] -= self.learningRate * error

    def train(self, inputsBatch, correctIndexes, epochs=10):
        """Train for a number of epochs """
        for _ in range(epochs):
            for inputs, correct in zip(inputsBatch, correctIndexes):
                self.step(inputs, correct)

## Function that will be used to create datasets

In [ ]:
def randomNumber(minimum=-10.0, maximum=10.0, decimals=2):
    return(round(random.uniform(minimum, maximum), decimals))

def createInputs(inputs=10):
    return([randomNumber() for _ in range(inputs)])

def createLayerData(size=5, inputs=10):
    weights = [[randomNumber(minimum=-1.0, maximum=1.0) for _ in range(inputs)] for _ in range(size)]
    biases = [randomNumber() for _ in range(size)]
    return(weights, biases)

## Now lets run a forward pass

In [232]:
random.seed(1)

inputsCount = 5
inputs = createInputs(inputsCount)
weights1, biases1 = createLayerData(3, inputsCount)
targets = [0,1,0]
layer1 = Layer(inputs, weights1, biases1, activationFunc=Activation.ReLU)
layer1Output = layer1.forward()
weights2, biases2 = createLayerData(3, len(layer1Output))  #< layer1Output is the input for the next layer
layer2 = Layer(layer1Output, weights2, biases2, activationFunc=Activation.ReLU)
layer2Output = layer2.forward()
softmaxOutput = Activation.Softmax.forward(layer2Output)
error = Loss.forward(softmaxOutput, targets)

## seeing model accuracy
accuracy = Accuracy_hard.calc(layer2Output, targets)    #< we want to maximize the 2nd output
print("INPUTS: ", inputs)
print("WEIGHTS: ", weights1)
print("BIASES: ", biases1)
print("RESULT: ", layer1Output)
print()
print("WEIGHTS: ", weights2)
print("BIASES: ", biases2)
print("RESULT: ", layer2Output)
print("SOFTMAX: ", softmaxOutput)
print()
print()
print("ERROR: ", error)
print("ACCURACY: ", accuracy)


INPUTS:  [-7.31, 6.95, 5.28, -4.9, -0.09]
WEIGHTS:  [[-0.1, 0.3, 0.58, -0.81, -0.94], [0.67, -0.13, 0.52, -1.0, -0.11], [0.44, -0.54, 0.89, 0.8, -0.94]]
BIASES:  [-9.49, 0.83, 8.78]
RESULT:  [0.44200000000000017, 2.6843, 2.6743999999999986]

WEIGHTS:  [[-0.24, -0.57, -0.16], [-0.94, -0.56, -0.12], [-0.01, -0.53, -0.54]]
BIASES:  [-5.62, -0.81, -4.2]
RESULT:  [0, 0, 0]
SOFTMAX:  [0.3333333333333333, 0.3333333333333333, 0.3333333333333333]


ERROR:  1.0986122886681098
ACCURACY:  0


### Now lets run a manual backward pass, calculating and storing every derivative, and then apply the chain rule ourself
* Later we can show how to improve this by passing the gradient to the next layer and allowing it to compute the chain rule itself

In [237]:
d_error  = Loss.backward(softmaxOutput, targets)
print(d_error)
d_softmax = Activation.Softmax.backward(layer2Output)
print(d_softmax)
d_layer2 = layer2.backward(layer1Output)
print(d_layer2)
d_layer1 = layer1.backward(inputs)
print(d_layer1)
d_layers = Mathlib.elementWiseMult(d_layer1*d_layer2)
d_softmax = Mathlib.dotVectorMatrix(d_error,d_softmax)
d_total = Mathlib.elementWiseMult(d_layers,d_softmax)
print(d_total)

[0.0, -3.0, 0.0]
[[0, 0, 0], [0, 0, 0], [0, 0, 0]]
[([-0.24, -0.57, -0.16], [0.44200000000000017, 2.6843, 2.6743999999999986], 1), ([-0.94, -0.56, -0.12], [0.44200000000000017, 2.6843, 2.6743999999999986], 1), ([-0.01, -0.53, -0.54], [0.44200000000000017, 2.6843, 2.6743999999999986], 1)]
[([-0.0, 0.0, 0.0, -0.0, -0.0], [-0.0, 0.0, 0.0, -0.0, -0.0], 0), ([0.67, -0.13, 0.52, -1.0, -0.11], [-7.31, 6.95, 5.28, -4.9, -0.09], 1), ([0.44, -0.54, 0.89, 0.8, -0.94], [-7.31, 6.95, 5.28, -4.9, -0.09], 1)]


TypeError: can't multiply sequence by non-int of type 'list'